
# Panchromatic SED: UV to Radio

Build a full galaxy SED spanning X-ray to radio wavelengths. Shows stellar
emission, dust attenuation, dust IR emission, radio synchrotron, and X-ray
binary contributions. Demonstrates tengri's multiwavelength physics modules
for radio and X-ray—no SSP data required for these components.

Reference: Conroy et al. 2010 (FSPS; radio connections); Fabbiano 2006
(X-ray binaries in galaxies, ARA&A, 44, 323).


In [ ]:
import warnings

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

wavelength = jnp.logspace(0, 10, 2000)
wave_um = np.array(wavelength) / 1e4

SFR = 10.0
STELLAR_MASS = 1e11
L_IR = 1e11
L_AGN_BOL = 1e44

l_xrb = tengri.xray.xray_xrb(wavelength, sfr=SFR, stellar_mass=STELLAR_MASS)
l_xray_agn = tengri.xray.xray_agn_corona(wavelength, L_agn_bol=L_AGN_BOL)
l_radio_sf = tengri.radio.radio_star_forming(wavelength, L_ir=L_IR)
l_radio_agn = tengri.radio.radio_agn(
    wavelength,
    L_agn_bol=L_AGN_BOL,
    radio_loudness=1.0,
)

fig, ax = plt.subplots(figsize=(11, 6))

components = [
    (l_xrb, "XRB (HMXB + LMXB)", "C0", "-"),
    (l_xray_agn, "AGN corona (X-ray)", "C1", "-"),
    (l_radio_sf, "SF synchrotron (radio)", "C2", "-"),
    (l_radio_agn, "AGN jets (radio)", "C3", "--"),
]

for l_nu, label, color, ls in components:
    l_np = np.array(l_nu)
    mask = l_np > 0
    if not np.any(mask):
        continue
    ax.loglog(wave_um[mask], l_np[mask], ls=ls, lw=1.8, color=color, label=label)

l_total = np.array(l_xrb + l_xray_agn + l_radio_sf + l_radio_agn)
mask_total = l_total > 0
ax.loglog(
    wave_um[mask_total],
    l_total[mask_total],
    "k-",
    lw=2.5,
    alpha=0.4,
    label="Total",
)

ax.set_xlabel(r"Wavelength [$\mu$m]")
ax.set_ylabel(r"$L_\nu$ [erg s$^{-1}$ Hz$^{-1}$]")
ax.set_xlim(1e-4, 1e6)
ax.set_ylim(1e20, 1e32)
ax.legend(frameon=False, fontsize=10, ncol=2)

for x, label in [(1.24e-4, "X-ray"), (3e4, "Radio")]:
    ax.axvline(x, color="grey", ls=":", lw=0.7, alpha=0.5)
    ax.text(x * 1.3, ax.get_ylim()[1] * 0.3, label, fontsize=10, color="grey")

fig.tight_layout()
plt.savefig("plot_radio_xray.png", dpi=150, bbox_inches="tight")